# Breast Cancer — Exploratory Data Analysis
Analisi esplorativa del dataset `breast_cancer` di scikit-learn per la predizione di cellule tumorali (maligno vs benigno).

In [ ]:
import sys
sys.path.insert(0, '..')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.datasets import load_breast_cancer
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler

sns.set_theme(style='whitegrid', palette='muted')
%matplotlib inline

## 1. Caricamento del dataset

In [ ]:
data = load_breast_cancer()
df = pd.DataFrame(data.data, columns=data.feature_names)
df['target'] = data.target
df['diagnosis'] = df['target'].map({0: 'Malignant', 1: 'Benign'})

print(f'Shape: {df.shape}')
print(f'Features: {len(data.feature_names)}')
print(f'\nTarget classes: {dict(zip(data.target_names, np.bincount(data.target)))}')
df.head()

## 2. Statistiche descrittive

In [ ]:
df.drop(columns=['target', 'diagnosis']).describe().T.style.background_gradient(cmap='Blues', subset=['mean', 'std'])

## 3. Distribuzione delle classi

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 4))

counts = df['diagnosis'].value_counts()
axes[0].bar(counts.index, counts.values, color=['#e74c3c', '#2ecc71'])
axes[0].set_title('Conteggio classi', fontweight='bold')
axes[0].set_ylabel('Campioni')
for i, v in enumerate(counts.values):
    axes[0].text(i, v + 3, str(v), ha='center', fontweight='bold')

axes[1].pie(counts.values, labels=counts.index, autopct='%1.1f%%',
            colors=['#e74c3c', '#2ecc71'], startangle=90)
axes[1].set_title('Proporzione classi', fontweight='bold')

plt.suptitle('Distribuzione Target', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

## 4. Distribuzioni delle feature principali

In [ ]:
features_mean = [c for c in df.columns if 'mean' in c]
fig, axes = plt.subplots(2, 5, figsize=(18, 8))
axes = axes.flatten()

for i, feat in enumerate(features_mean):
    for label, color in [('Malignant', '#e74c3c'), ('Benign', '#2ecc71')]:
        axes[i].hist(df[df['diagnosis'] == label][feat], bins=30,
                     alpha=0.6, label=label, color=color)
    axes[i].set_title(feat.replace(' (mean)', ''), fontsize=8)
    axes[i].legend(fontsize=7)

plt.suptitle('Distribuzioni feature "mean" per classe', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

## 5. Boxplot comparativo

In [ ]:
top5 = ['mean radius', 'mean texture', 'mean perimeter', 'mean area', 'mean concavity']
fig, axes = plt.subplots(1, 5, figsize=(16, 5))

for ax, feat in zip(axes, top5):
    df.boxplot(column=feat, by='diagnosis', ax=ax,
               boxprops=dict(color='#2c3e50'),
               medianprops=dict(color='#e74c3c', linewidth=2))
    ax.set_title(feat.replace('mean ', ''), fontsize=9)
    ax.set_xlabel('')

plt.suptitle('Boxplot feature principali per classe', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

## 6. Matrice di correlazione

In [ ]:
X = df.drop(columns=['target', 'diagnosis'])
corr = X.corr()
mask = np.triu(np.ones_like(corr, dtype=bool))

fig, ax = plt.subplots(figsize=(15, 12))
sns.heatmap(corr, mask=mask, annot=False, cmap='coolwarm', center=0,
            linewidths=0.3, ax=ax, cbar_kws={'shrink': 0.8})
ax.set_title('Feature Correlation Matrix', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

# Top correlazioni con il target
target_corr = X.corrwith(df['target']).abs().sort_values(ascending=False)
print('\nTop 10 feature per correlazione con il target:')
print(target_corr.head(10).to_string())

## 7. PCA — Visualizzazione 2D

In [ ]:
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

pca = PCA(n_components=2, random_state=42)
X_pca = pca.fit_transform(X_scaled)

fig, ax = plt.subplots(figsize=(8, 6))
colors = {0: '#e74c3c', 1: '#2ecc71'}
labels = {0: 'Malignant', 1: 'Benign'}
for cls in [0, 1]:
    mask = df['target'] == cls
    ax.scatter(X_pca[mask, 0], X_pca[mask, 1],
               c=colors[cls], label=labels[cls], alpha=0.6, s=30)

ax.set_xlabel(f'PC1 ({pca.explained_variance_ratio_[0]*100:.1f}% varianza)')
ax.set_ylabel(f'PC2 ({pca.explained_variance_ratio_[1]*100:.1f}% varianza)')
ax.set_title('PCA — Proiezione 2D', fontsize=13, fontweight='bold')
ax.legend()
plt.tight_layout()
plt.show()

total_var = pca.explained_variance_ratio_.sum()
print(f'Varianza spiegata dalle prime 2 componenti: {total_var*100:.1f}%')

## 8. Scree Plot — Varianza spiegata dalla PCA

In [ ]:
pca_full = PCA(random_state=42).fit(X_scaled)
cumvar = np.cumsum(pca_full.explained_variance_ratio_)

fig, ax = plt.subplots(figsize=(9, 5))
ax.bar(range(1, len(pca_full.explained_variance_ratio_) + 1),
       pca_full.explained_variance_ratio_, alpha=0.7, color='#3498db', label='Individuale')
ax.plot(range(1, len(cumvar) + 1), cumvar, 'o-', color='#e74c3c', label='Cumulativa')
ax.axhline(0.95, color='gray', linestyle='--', label='95% soglia')
n95 = np.argmax(cumvar >= 0.95) + 1
ax.axvline(n95, color='orange', linestyle='--', label=f'{n95} componenti → 95%')
ax.set_xlabel('Componente Principale')
ax.set_ylabel('Varianza Spiegata')
ax.set_title('Scree Plot', fontsize=13, fontweight='bold')
ax.legend()
plt.tight_layout()
plt.show()
print(f'{n95} componenti principali spiegano il 95% della varianza')

## 9. Valori mancanti e outlier

In [ ]:
print('Valori mancanti:')
print(X.isnull().sum().sum(), 'valori mancanti totali\n')

# Outlier detection via IQR
Q1, Q3 = X.quantile(0.25), X.quantile(0.75)
IQR = Q3 - Q1
outliers = ((X < Q1 - 1.5 * IQR) | (X > Q3 + 1.5 * IQR)).sum()
print('Outlier per feature (metodo IQR):')
print(outliers.sort_values(ascending=False).head(10).to_string())

## 10. Riepilogo

| Aspetto | Dettaglio |
|---|---|
| Campioni totali | 569 |
| Feature | 30 (mean, SE, worst per 10 misure) |
| Classe 0 (Maligno) | 212 (37.3%) |
| Classe 1 (Benigno) | 357 (62.7%) |
| Valori mancanti | Nessuno |
| Feature più discriminante | `mean concave points`, `mean perimeter`, `mean radius` |
| PC per 95% varianza | ~10 componenti |

> **Prossimo passo:** eseguire `python train.py` per addestrare e confrontare i modelli.